# PriceFM_Plus tutorial

This notebook downloads the public [European_Foundation_Model_DAM](https://huggingface.co/datasets/RunyaoYu/European_Foundation_Model_DAM) release and checks its data, trained weights, forecasts, Table 3, and figures. It reruns a small held-out inference sample for each trained model. Saved outputs beneath the cells come from a verified run. Downloads are several gigabytes.

Use Python 3.10 or newer. From this clone, install `python -m pip install -e ".[train,figures]"` and restart the kernel. The published wheel can also be installed directly with `python -m pip install "PriceFM_Plus[train,figures] @ https://huggingface.co/datasets/RunyaoYu/European_Foundation_Model_DAM/resolve/main/pricefm_plus-0.1.0-py3-none-any.whl"`.


## 1. Choose an output directory

Downloads and results are written outside the Git clone. Set `PRICEFM_PLUS_TUTORIAL_WORKDIR` to choose another directory.


In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import PriceFM_Plus
from importlib.metadata import version

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

launch_dir = Path.cwd()
workdir = Path(os.environ.get("PRICEFM_PLUS_TUTORIAL_WORKDIR", Path.home() / "PriceFM_Plus_demo"))
workdir.mkdir(parents=True, exist_ok=True)
os.chdir(workdir)
print("Output directory:", workdir.name)
print("Installed package version:", version("PriceFM_Plus"))
print("Public API:", ", ".join(PriceFM_Plus.__all__))


Output directory: tutorial_run
Installed package version: 0.1.0
Public API: add_data, customize, download_data, download_forecasts, download_weights, reproduce_figures, reproduce_forecasts, reproduce_metrics


## 2. Download raw and processed data

Both archives come from Hugging Face. The package verifies ZIP CRC before extracting into `Data/`.


In [2]:
for kind in ("raw", "processed"):
    root = PriceFM_Plus.download_data(kind)
    folder = root / ("Raw" if kind == "raw" else "Processed")
    print(f"{kind}: {sum(p.is_file() for p in folder.rglob('*')):,} files in {folder.relative_to(Path.cwd())}")
assert (Path("Data/Config/active_regions.json")).is_file()
print("Data download and extraction verified")


raw: 5,644 files in Data/Raw
processed: 1,426 files in Data/Processed
Data download and extraction verified


## 3. Download trained model weights

These are the archived MLP, LSTM, Transformer, and PriceFM_Plus weights. Foundation-model checkpoints are not required for the metrics and figures below.


In [3]:
trained_models = ("MLP", "LSTM", "Transformer", "PriceFM_Plus")
for model in trained_models:
    root = PriceFM_Plus.download_weights(model)
    count = sum(p.is_file() for p in (root / model).rglob("*"))
    assert count > 0
    print(f"{model}: {count:,} weight files")


MLP: 1,440 weight files
LSTM: 1,440 weight files
Transformer: 1,440 weight files
PriceFM_Plus: 3,360 weight files


## 4. Rerun one held-out forecast per trained model

This uses AT, the Realistic feature case, and test fold 1. The cell moves each generated forecast to `Inferred/` before archived forecasts are downloaded, so the public forecasts are never overwritten. A GPU is helpful for this step.


In [4]:
for model in trained_models:
    relative = Path(model) / "Realistic" / "Joint" / ("L2" if model == "PriceFM_Plus" else "Include_2022") / "fold1.npz"
    target = Path("Inferred") / relative
    if not target.is_file():
        generated = PriceFM_Plus.reproduce_forecasts(
            model, regions=["AT"], cases=["Realistic"], folds=(1,), batch_days=1
        )[0]
        target.parent.mkdir(parents=True, exist_ok=True)
        generated.replace(target)
    with np.load(target, allow_pickle=False) as z:
        assert z["regions"].astype(str).tolist() == ["AT"]
        print(f"{model}: {len(z['region_index']):,} AT forecast slots; {target}")


MLP: 8,640 AT forecast slots; Inferred\MLP\Realistic\Joint\Include_2022\fold1.npz
LSTM: 8,640 AT forecast slots; Inferred\LSTM\Realistic\Joint\Include_2022\fold1.npz
Transformer: 8,640 AT forecast slots; Inferred\Transformer\Realistic\Joint\Include_2022\fold1.npz
PriceFM_Plus: 8,640 AT forecast slots; Inferred\PriceFM_Plus\Realistic\Joint\L2\fold1.npz


## 5. Download and inspect archived forecasts

This downloads all four trained-model archives, four foundation-model forecast archives, and Persistence. Forecast NPZ files contain dates, region labels, row indices, time slots, and native-price quantile predictions. They contain no training histories or validation losses. The cell compares the inferred AT sample with its archived counterpart.


In [5]:
forecast_models = ("Persistence", *trained_models, "TabPFN_TS_3", "TiRex_2", "TimesFM_3", "Chronos_2")
for model in forecast_models:
    root = PriceFM_Plus.download_forecasts(model)
    print(f"{model}: {sum(p.is_file() for p in (root / model).rglob('*'))} forecast files")

for model in trained_models:
    relative = Path(model) / "Realistic" / "Joint" / ("L2" if model == "PriceFM_Plus" else "Include_2022") / "fold1.npz"
    with np.load(Path("Inferred") / relative, allow_pickle=False) as inferred, np.load(Path("Forecasts") / relative, allow_pickle=False) as saved:
        index = np.where(saved["regions"].astype(str) == "AT")[0][0]
        actual = inferred["prediction_EUR_per_MWh"]
        archived = saved["prediction_EUR_per_MWh"][saved["region_index"] == index]
        assert actual.shape == archived.shape
        error = np.abs(actual - archived)
        print(f"{model}: {len(actual):,} matching slots; mean |difference| = {error.mean():.4f}, max = {error.max():.4f} EUR/MWh")

with np.load(Path("Forecasts/PriceFM_Plus/Realistic/Joint/L2/fold1.npz"), allow_pickle=False) as z:
    print("Archived NPZ keys:", ", ".join(z.files))
    print("Archived prediction shape:", z["prediction_EUR_per_MWh"].shape)


Persistence: 9 forecast files
MLP: 27 forecast files
LSTM: 27 forecast files
Transformer: 27 forecast files
PriceFM_Plus: 63 forecast files
TabPFN_TS_3: 9 forecast files
TiRex_2: 9 forecast files
TimesFM_3: 9 forecast files
Chronos_2: 9 forecast files
MLP: 8,640 matching slots; mean |difference| = 0.0207, max = 0.1622 EUR/MWh
LSTM: 8,640 matching slots; mean |difference| = 0.0238, max = 0.1284 EUR/MWh
Transformer: 8,640 matching slots; mean |difference| = 0.0143, max = 0.1186 EUR/MWh
PriceFM_Plus: 8,640 matching slots; mean |difference| = 0.0230, max = 0.1911 EUR/MWh
Archived NPZ keys: dates, regions, region_index, day_index, slot_utc, prediction_EUR_per_MWh
Archived prediction shape: (332784, 3)


## 6. Reproduce Table 3

`reproduce_metrics()` scores saved forecasts against observed prices in downloaded processed data. It writes `Metrics/table3.csv` with pinball loss, interval score, MAE, and RMSE. The reference CSV beside this notebook lets us check every reported value.


In [6]:
table3 = PriceFM_Plus.reproduce_metrics()
reference_candidates = (launch_dir / "tutorials" / "reference_table3.csv", launch_dir / "reference_table3.csv")
reference_path = next((p for p in reference_candidates if p.is_file()), None)
if reference_path:
    reference = pd.read_csv(reference_path)
    merged = table3.merge(reference, on=["model", "feature_case"], suffixes=("_new", "_ref"), validate="one_to_one")
    assert len(merged) == len(reference) == 33
    columns = ("Pinball", "Interval Score", "MAE", "RMSE")
    max_difference = max(np.abs(merged[f"{column}_new"] - merged[f"{column}_ref"]).max() for column in columns)
    assert max_difference <= 1e-6
    print(f"Verified all {len(merged) * len(columns)} reported metrics; maximum absolute difference {max_difference:.2g}")
else:
    print("Reference CSV unavailable in this launch directory; reproduced table saved to Metrics/table3.csv")
print("Saved:", Path("Metrics/table3.csv"))


        model feature_case  Pinball  Interval Score   MAE  RMSE  region_days
Persistence_1    Realistic    10.01          155.20 29.03 45.47        10920
Persistence_1          TSO    10.01          155.20 29.03 45.47        10920
Persistence_1       Oracle    10.01          155.20 29.03 45.47        10920
Persistence_2    Realistic    10.00          151.13 29.77 45.69        10920
Persistence_2          TSO    10.00          151.13 29.77 45.69        10920
Persistence_2       Oracle    10.00          151.13 29.77 45.69        10920
Persistence_3    Realistic     9.18          137.31 27.65 43.40        10920
Persistence_3          TSO     9.18          137.31 27.65 43.40        10920
Persistence_3       Oracle     9.18          137.31 27.65 43.40        10920
          MLP    Realistic     7.44          112.12 22.20 35.41        10920
          MLP          TSO     6.75          101.98 20.11 32.74        10920
          MLP       Oracle     6.68          100.82 19.89 31.92        10920

## 7. Regenerate the paper figures

This downloads the released plotting source and figure-input data, then creates five grouped figures and four ablation panels in `Figures/`.


In [7]:
figure_dir = PriceFM_Plus.reproduce_figures()
pdfs = sorted(figure_dir.glob("*.pdf"))
assert len(pdfs) == 9
for pdf in pdfs:
    assert pdf.stat().st_size > 0
    print(pdf.name, f"({pdf.stat().st_size / 1024:.0f} KiB)")
print("Verified 9 regenerated PDFs in", Path("Figures"))


abl_figure.pdf (152 KiB)
cr_figure.pdf (704 KiB)
eda_figure.pdf (15538 KiB)
fm_figure.pdf (12765 KiB)
jt_figure.pdf (12598 KiB)
model_ablation_no_lag_lead.pdf (29 KiB)
model_ablation_no_region_encoding.pdf (31 KiB)
model_ablation_no_temporal_mixing.pdf (30 KiB)
model_ablation_single_chronological_branch.pdf (29 KiB)
Verified 9 regenerated PDFs in Figures


## 8. Extend data with ENTSO-E

Set `ENTSOE_API_TOKEN` in your environment, then choose new dates for your own update. End dates are exclusive. The example updates one previously archived day, fetches missing raw records, and verifies all three processed variants. It uses no credential stored in the notebook.


In [8]:
from contextlib import redirect_stdout
from io import StringIO

token = os.environ.get("ENTSOE_API_TOKEN")
if token:
    with redirect_stdout(StringIO()):
        updates = PriceFM_Plus.add_data("2026-09-01", "2026-09-02", token)
    download = updates["copy"]["download"]
    print(f"Raw request intervals: {download['planned']}; failed: {download.get('failed', 0)}")
    for fill, result in updates.items():
        print(f"{fill}: {result['processed_series']} processed series; {result['checked_series']} checked series")
else:
    print("Set ENTSOE_API_TOKEN to run this example with your own credentials.")


Raw request intervals: 147; failed: 0
copy: 520 processed series; 520 checked series
interp: 520 processed series; 0 checked series
none: 520 processed series; 0 checked series


## 9. Optional: train a custom model

This one-region, one-epoch example demonstrates `customize()`. It is intentionally not executed in the verified tutorial run. The six dates give train, validation, and test start/end boundaries; end dates are exclusive. Omitted hyperparameters use the paper settings.


In [9]:
RUN_CUSTOM_TRAINING = False
if RUN_CUSTOM_TRAINING:
    result = PriceFM_Plus.customize(
        dates=["2025-01-01", "2025-01-14", "2025-01-14", "2025-01-16", "2025-01-16", "2025-01-18"],
        feature_case="Realistic", regions=["AT"],
        hyperparams={"epochs": 1, "microbatch_size": 64, "batch_size": 64},
    )
    print(result)
else:
    print("Custom training example retained but not run.")


Custom training example retained but not run.


## 10. Optional: rerun a foundation model

Install its model-specific extra first. If its checkpoint is absent, the package asks before downloading it. This example is intentionally not executed in the verified tutorial run.


In [10]:
RUN_FOUNDATION_INFERENCE = False
if RUN_FOUNDATION_INFERENCE:
    paths = PriceFM_Plus.reproduce_forecasts("Chronos_2", regions=["AT"], cases=["Realistic"], folds=(1,))
    print("Generated:", *paths, sep="\n")
else:
    print("Foundation-model inference example retained but not run.")


Foundation-model inference example retained but not run.
